# Day 3 · Lab 6 — Finding Attacks With No Labels At All

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyberirishman/5-day-AI-Cyber/Day3_Anomaly_Lab_v1.ipynb)

### What you are about to do

Yesterday you built a Decision Tree that found attacks by being **told** which
logins were attacks. Somebody handed you a column called `Is Attack IP` and your
model learned to copy it. Then we showed you two uncomfortable things:

1. That column was a **reputation blocklist**, not a record of what happened.
2. Your model had really only learned one thing — *"is this login from outside Norway?"* —
   and a €3 Norwegian proxy walked straight past it.

**In the real world nobody hands you that column.** Nobody has labelled your company's
logs. So today you build a detector that is never told the answer. It does not ask
*"is this an attack?"* It asks a different question:

> ### "Is this login WEIRD compared to everything else?"

That is called **anomaly detection**, and it is how detection actually works in a SOC.

### The plan

| Step | What happens |
|---|---|
| 1–3 | Load 11,494 real logins and **look at them** — normal ones, noise, and three real attacks |
| 4 | See how much of the truth the old label actually covered |
| 5 | Build 8 features — and fall into a data-leakage trap on purpose, so you can spot it later |
| 6 | Detector 1: **Isolation Forest** (an idea you already half know) |
| 7 | Detector 2: a **neural network** — your first one |
| 8–10 | Reveal the answer key, compare the two, and find out where both of them fail |

**Runtime:** Google Colab, free tier, CPU. Nothing to install. Roughly 35 minutes.
**Click `Runtime -> Run all` if you get lost, then come back to wherever you were.**

---
## STEP 0 — The tools we are using

Four libraries. You do not need to install anything: Colab already has all four.

**Two of them are data handling. One is CLASSIC MACHINE LEARNING. One is a NEURAL-NETWORK
framework. Those last two are not the same kind of thing at all**, and knowing which is which
is half of understanding this lab:

| Library | Category | What it is | Used for |
|---|---|---|---|
| **pandas** | data handling | Spreadsheets for Python. A `DataFrame` is a table with named columns. | Loading and slicing the log |
| **numpy** | data handling | Fast maths on big lists of numbers. | Arithmetic on the feature table |
| **matplotlib** | data handling | The standard plotting library. | The two charts at the end |
| **scikit-learn** | 🔧 **CLASSIC MACHINE LEARNING** | The toolbox for the non-neural models — decision trees, random forests, clustering, Isolation Forest. Yesterday's Decision Tree came from here too. **There is no neural network anywhere in scikit-learn.** | **Detector 1** (Isolation Forest) and rescaling the numbers |
| **tensorflow** + **keras** | 🧠 **DEEP LEARNING FRAMEWORK** | A completely different kind of library: it builds neural networks out of layers of artificial neurons and trains them. TensorFlow is Google's engine; Keras is the friendly front door on top of it. The same family of tool that trains the large language models from Day 1. | **Detector 2** (the autoencoder) |

Why two libraries? Because the two detectors are two different *kinds* of model. Detector 1 is
classic ML — no neurons, no layers, no training loop. Detector 2 is a real (if tiny) neural
network. **Watching them compete is part of the point of this lab.**

Run the cell below. It should print four version numbers and the word **READY**.

In [ ]:
# ============================================================
# STEP 0 — IMPORT THE TOOLS
# Nothing here touches the data. We are just loading libraries
# and printing their versions so that if anything goes wrong
# later, we know exactly which versions we were running.
# ============================================================

import pandas as pd                    # tables
import numpy as np                     # numbers
import matplotlib.pyplot as plt        # charts

# ------------------------------------------------------------
# CLASSIC MACHINE LEARNING  —  scikit-learn
# The non-neural toolbox: decision trees, random forests,
# clustering, Isolation Forest. Yesterday's Decision Tree came
# from here too. There is NOT ONE neural network in scikit-learn.
# ------------------------------------------------------------
from sklearn.ensemble import IsolationForest      # <<< DETECTOR 1 — classic ML
from sklearn.preprocessing import StandardScaler  # rescales columns (not a model)

# ------------------------------------------------------------
# NEURAL NETWORKS  —  TensorFlow + Keras
# A DIFFERENT KIND OF LIBRARY ENTIRELY. This one builds networks
# out of layers of artificial neurons and trains them. TensorFlow
# is Google's engine; Keras is the friendly front door on top of
# it. Same family of tool that trains the LLMs from Day 1.
# ------------------------------------------------------------
import tensorflow as tf                # the engine
from tensorflow import keras           # <<< DETECTOR 2 gets built with this

print("DATA HANDLING")
print("  pandas        ", pd.__version__)
print("  numpy         ", np.__version__)
print()
print("CLASSIC MACHINE LEARNING   -> detector 1")
import sklearn; print("  scikit-learn  ", sklearn.__version__)
print()
print("NEURAL NETWORKS            -> detector 2")
print("  tensorflow    ", tf.__version__)
print("  keras         ", keras.__version__)
print()
print("READY - two different families of model, one dataset.")

---
## STEP 1 — Load the log

The file is `day3_auth_enriched_v1.csv`, sitting in the course GitHub repository.
The next cell downloads it straight from there, so **there is nothing for you to
upload and nothing to download by hand**.

### What is in this file

It is a **web login log** — one row for every time somebody tried to sign in to a
Norwegian company's website over 7 days in June. It is built from a real, published
research dataset (the RBA dataset, Wiefling et al.), with extra accounts and extra
attacks added for this lab. Every row is one login attempt:

| Column | Means |
|---|---|
| `Login Timestamp` | exactly when it happened, to the millisecond |
| `Username` | which account was being logged in to |
| `IP Address` | the internet address it came from |
| `Country` / `Region` / `City` | where in the world that address is |
| `ASN` | which network owns that address (a phone network, an office, a hosting company) |
| `User Agent String` | what the software claimed to be — a browser, a phone, or a script |
| `Device Type` | desktop / mobile / tablet / **bot** |
| `Login Successful` | **True = they got in. False = wrong password.** |
| `Is Attack IP` | yesterday's blocklist label. **We are not allowed to use this today.** |
| `Is Account Takeover` | the answer key for the worst case. **Also off limits today.** |

> ⚠ **The two label columns are the ones we are refusing to use.** They stay in the
> file so we can grade ourselves at the very end — like sealing the answers in an
> envelope before the exam.

In [ ]:
# ============================================================
# STEP 1 — LOAD THE DATA
# pd.read_csv() can read straight from a web address.
# ============================================================

DATA_URL = "https://raw.githubusercontent.com/cyberirishman/5-day-AI-Cyber/main/day3_lab6/day3_auth_enriched_v1.csv"

logs = pd.read_csv(DATA_URL)

# Turn the timestamp text into a real date/time object so Python can do
# arithmetic on it (subtract two of them, ask for the hour, and so on).
logs["Login Timestamp"] = pd.to_datetime(logs["Login Timestamp"])

# Sort oldest-first. This matters a LOT in step 5 — hold that thought.
logs = logs.sort_values("Login Timestamp").reset_index(drop=True)

print("rows loaded :", f"{len(logs):,}")
print("columns     :", len(logs.columns))
print("date range  :", logs["Login Timestamp"].min(), "->", logs["Login Timestamp"].max())
print("accounts    :", f"{logs.Username.nunique():,}")
print()
print("STOP AND CHECK: it should say 11,494 rows and 16 columns.")

---
## STEP 2 — Look at the data before you model it

**This is the step everybody skips and everybody regrets.** You cannot build a
detector for behaviour you have never looked at.

We are going to print real lines from the log in a shortened format, so they fit on
one line each:

```
DATE TIME        ACCOUNT       IP ADDRESS      CC  CLIENT     RESULT
```

First: what does an ordinary, boring, innocent login look like?

In [ ]:
# ============================================================
# STEP 2a — A HELPER FOR PRINTING LOG LINES
# This is just formatting. It does no analysis. It squeezes the
# useful columns into one readable line so we can eyeball them.
# ============================================================

def show(rows, title):
    print("=" * 78)
    print(title)
    print("=" * 78)
    for _, r in rows.iterrows():
        ua = str(r["User Agent String"])
        # Shorten the giant user-agent string down to something readable
        if "Zippp" in ua:          client = "BOT/Zippp"
        elif "Go-http" in ua:      client = "Go-script"
        elif "python-requests" in ua: client = "py-script"
        elif "curl" in ua:         client = "curl"
        elif "hydra" in ua:        client = "hydra"
        elif "Chrome" in ua:       client = "Chrome"
        elif "Safari" in ua:       client = "Safari"
        else:                      client = ua[:9]
        print(f'{r["Login Timestamp"]:%m-%d %H:%M:%S}  {r["Username"]:<12} '
              f'{r["IP Address"]:<15} {r["Country"]:<3} {client:<10} '
              f'{"GOT IN" if r["Login Successful"] else "FAILED"}')
    print()


# Find an ordinary account: several logins, all from Norway, all successful.
tally = logs.assign(
    is_norway=(logs.Country == "NO").astype(int),
    is_ok=logs["Login Successful"].astype(int),
).groupby("Username")[["is_norway", "is_ok"]].agg(["size", "sum"])

quiet_lives = tally[(tally[("is_norway", "size")] >= 5)
                    & (tally[("is_norway", "sum")] == tally[("is_norway", "size")])
                    & (tally[("is_ok", "sum")] == tally[("is_ok", "size")])]
who = sorted(quiet_lives.index)[0]

show(logs[logs.Username == who].head(6),
     f"NORMAL — {who}: their own country, their own browser, every time")

print("Same account. Same country. Same browser. The address changes a little")
print("because home broadband does that. Nothing here needs investigating.")

### Now the noise

Most failed logins are not attacks. They are **people typing their own password wrong.**
One failure, one account, and you never see them again. There are **thousands** of these.

If your detector flags these, it is worthless — your analysts will drown.

In [ ]:
# ============================================================
# STEP 2b — THE NOISE: ordinary human mistakes
# Find accounts that appear exactly ONCE in the whole log,
# and that one appearance was a failure. Fat fingers.
# ============================================================

appearances = logs.Username.value_counts()
seen_once = appearances[appearances == 1].index
fat_fingers = logs[(logs.Username.isin(seen_once)) & (~logs["Login Successful"])]

show(fat_fingers.head(4), "NOISE — a single mistyped password, then nothing")

print(f"There are {len(fat_fingers):,} logins like this in the file.")
print(f"That is {len(fat_fingers) / (~logs['Login Successful']).sum():.0%} of ALL the failures in the log.")
print()
print("REMEMBER THIS NUMBER. A good detector must IGNORE all of them.")

---
## STEP 3 — The three attacks, in the actual log lines

There are three genuinely malicious patterns hiding in this file. Read each one and
ask yourself: *what would make this stand out to a computer?*

In [ ]:
# ============================================================
# STEP 3a — ATTACK 1: CREDENTIAL STUFFING
# The attacker has a list of stolen passwords from some other
# breached website, and tries them all against ONE account,
# spreading the attempts over hundreds of different addresses
# so that no single address looks busy.
# ============================================================

stuffing = logs[logs.Username == "user_a44feb"].sort_values("Login Timestamp")
show(stuffing.iloc[5:11], "ATTACK 1 — CREDENTIAL STUFFING against user_a44feb")

print(f"attempts      : {len(stuffing)}")
print(f"unique IPs    : {stuffing['IP Address'].nunique()}")
print(f"countries     : {stuffing.Country.nunique()}")
print(f"ever got in?  : {stuffing['Login Successful'].sum()} times")
print()
print("WHAT MAKES IT WEIRD: nine seconds apart, from a different continent each")
print("time. No human logs in like that. But notice - no SINGLE line looks strange.")
print()

# ------------------------------------------------------------
# THIS IS NOT THE ONLY STUFFED ACCOUNT.
# There is a second, much quieter one - and quiet is exactly
# what makes an attack hard to find. Both are listed here so
# you can go and read the other one yourself.
# ------------------------------------------------------------
print("=" * 78)
print("TWO accounts in this file are hammered and never once get in:")
print("=" * 78)
for who in ["user_a44feb", "user_fd805b"]:
    g = logs[logs.Username == who]
    print(f"  {who}  ->  {len(g):>3} attempts  ·  {g['IP Address'].nunique():>3} addresses"
          f"  ·  {g.Country.nunique():>2} countries  ·  {g['Login Successful'].sum()} successes")
print()
print("user_fd805b is the more realistic one: 34 attempts, only 3 addresses, all in")
print("ONE country. Nothing about it shouts. Go and look at it for yourself:")
print()
print('    logs[logs.Username == "user_fd805b"]')

In [ ]:
# ============================================================
# STEP 3b — ATTACK 2: BOT BRUTE FORCE
# Automated software hammering an account from a small block of
# addresses. It does not even hide what it is: it puts its own
# name in the user-agent string.
# ============================================================

swarm = logs[logs["User Agent String"].str.contains("Zippp", na=False)]
swarm = swarm.sort_values("Login Timestamp")
show(swarm.head(5), "ATTACK 2 — AUTOMATED BRUTE FORCE (a bot)")

burst = swarm["Login Timestamp"].max() - swarm["Login Timestamp"].min()
print(f"attempts      : {len(swarm)}")
print(f"unique IPs    : {swarm['IP Address'].nunique()}  (all inside 10.3.205.x - one block)")
print(f"whole burst   : {burst}")
print(f"success rate  : {swarm['Login Successful'].mean():.0%}")
print()
print("WHAT MAKES IT WEIRD: a few seconds apart, never once succeeds, and it does")
print("not even hide - Device Type says 'bot' and the user-agent gives its own")
print("project URL. Easy to catch. Real attackers are not always this polite.")
print()

# ------------------------------------------------------------
# HOW MANY ACCOUNTS IS IT ATTACKING?
# Most people assume a brute-force tool sprays hundreds of
# accounts. This one does not. Count them and see.
# ------------------------------------------------------------
bots = logs[logs["Device Type"] == "bot"]
print("=" * 78)
print(f"ALL {len(bots)} automated rows in this file target exactly "
      f"{bots.Username.nunique()} account: {', '.join(bots.Username.unique())}")
print("=" * 78)
print("That is the SAME account attack 1 is stuffing. One target, hit two ways.")
print()

# ...but it is not ONE bot. Give each client a readable name first: the raw
# user-agent strings vary even within one tool, and two unrelated bots both
# start with "Mozilla", so grouping on the raw string miscounts them.
def bot_name(ua):
    ua = str(ua)
    for needle, label in [("Zippp",      "ZipppBot"),
                          ("startmebot", "startmebot"),
                          ("MetaJobBot", "MetaJobBot"),
                          ("ZoomBot",    "ZoomBot / Linkbot"),
                          ("NaverRobot", "dloader / NaverRobot")]:
        if needle.lower() in ua.lower():
            return label
    return ua[:24]

bots = bots.assign(client=bots["User Agent String"].map(bot_name))

print(f"It is not one bot either - it is {bots.client.nunique()} different clients:")
print()
for client, g in sorted(bots.groupby("client"), key=lambda kv: -len(kv[1])):
    print(f"  {client:<24} {len(g):>3} rows  ·  {g['IP Address'].nunique():>2} addresses"
          f"  ·  {g.Country.nunique()} countries  ·  {g['Login Successful'].sum()} successes")
print()
print("NOW READ THE FOUR SMALL ONES AGAIN - six rows between them. A search")
print("crawler, a Naver download robot, a link checker and a job scraper. Those")
print("are almost certainly NOT attacking anything: they crawled the login page")
print("and failed, which is what crawlers do all day.")
print()
print("But the answer key at the end of this lab counts them as attack rows,")
print("because that key was written per-ACCOUNT: every row belonging to an")
print("attacked account gets called an attack.")
print()
print("That is a real labelling decision, made by a human, and here are six rows")
print("where it is arguably wrong. It is the same question from earlier today:")
print("WHO DECIDED WHAT COUNTS AS AN ATTACK?")
print()
print("One more thing, before you trust 'it looks like a script' as a signal.")
print("Count the ATTACKER TOOLING - the clients a person never types with:")
print()
tooling = logs[logs["User Agent String"].str.contains(
    "Go-http|python-requests|curl/|hydra", case=False, na=False)]
print(f"  {len(tooling)} rows across {tooling.Username.nunique()} different accounts.")
print()
print("So a scripted client is not rare, and it is not unique to this attack.")
print("It is a SIGNAL. It is never a verdict.")

In [ ]:
# ============================================================
# STEP 3c — ATTACK 3: ACCOUNT TAKEOVER — and this one WORKS
#
# This is the one that costs money: someone guesses the password
# and GETS IN. We print this account's ENTIRE history - all 13
# rows, nothing hidden - because the whole story is in the order.
# ============================================================

victim = logs[logs.Username == "user_c99b98"].sort_values("Login Timestamp")
show(victim, f"ATTACK 3 — ACCOUNT TAKEOVER of user_c99b98  ({len(victim)} rows, all of them)")

# Split the account's own history from the attacker's burst, so we can
# measure the burst instead of guessing at it.
theirs  = victim[victim.Country == "NO"]
attack  = victim[victim.Country != "NO"]
fails   = attack[~attack["Login Successful"]]
seconds = (fails["Login Timestamp"].max() - fails["Login Timestamp"].min()).total_seconds()

own_span = theirs["Login Timestamp"].max() - theirs["Login Timestamp"].min()
quiet    = attack["Login Timestamp"].min() - theirs["Login Timestamp"].max()

print("THE SHAPE, MEASURED FROM THE ROWS ABOVE:")
print(f"  rows in total          : {len(victim)}")
print(f"  the real person's own  : {len(theirs)} logins, both successful, "
      f"{own_span.days} days apart")
print(f"  then nothing for       : {quiet.days} days - the account just sits there")
print(f"  the attacker's failures: {len(fails)}")
print(f"  ...spread over         : {seconds:.0f} seconds  (not minutes. seconds.)")
print(f"  the attacker's success : {int(attack['Login Successful'].sum())}  <-- the breach")
print(f"  successes in total     : {int(victim['Login Successful'].sum())}  "
      f"- 2 earned, 1 stolen")
print()

# ------------------------------------------------------------
# "A NORWEGIAN ADDRESS" - HOW DO WE ACTUALLY KNOW THAT?
# Not from the IP address itself. Two other columns tell us.
# ------------------------------------------------------------
print("=" * 78)
print("WHERE DOES 'NORWEGIAN' COME FROM? TWO COLUMNS, NOT THE IP ADDRESS.")
print("=" * 78)
print("1) The 'Country' column - printed as CC in the table above. It is a")
print("   two-letter country code the service worked out by geolocating the")
print("   address:  NO = Norway,  DE = Germany.  That is the only reason we")
print("   can call one 'home' and the other 'abroad'.")
print()
print("2) The 'ASN' column - which network owns the address. Look:")
print()
for label, rows in [("the real person", theirs), ("the attacker", attack)]:
    for asn, g in rows.groupby("ASN"):
        nets = sorted({".".join(ip.split(".")[:2]) + ".x.x" for ip in g["IP Address"]})
        print(f"   {label:<16} ASN {asn:<7} {g.Country.iloc[0]}   "
              f"{len(g):>2} rows   {', '.join(nets)}")
print()
print("   Every single login this person made came from 84.209.x.x on ONE network.")
print("   The attacker is ONE address on a completely different network.")
print("   Neither the country nor the network is in the IP number - you need the")
print("   Country and ASN columns to see it. (The addresses in this dataset are")
print("   pseudonymised by its authors, so do not go looking them up.)")
print()

# ------------------------------------------------------------
# HOW MANY TAKEOVERS ARE THERE, AND WHICH ONES ARE REAL?
# ------------------------------------------------------------
print("=" * 78)
print("THERE ARE MORE THAN ONE. THREE OF THEM ARE REAL.")
print("=" * 78)
takeovers = logs[logs["Is Account Takeover"]]
print(f"This file contains {len(takeovers)} successful takeovers out of "
      f"{len(logs):,} rows - {len(takeovers)/len(logs):.2%} of the data.")
print()
print("THREE of them are untouched rows from the original published research")
print("dataset. They are the real thing, and you can read all three right now:")
print()
ORIGINALS = ["user_840bfc", "user_a4549c", "user_c99b98"]
for who in ORIGINALS:
    r = takeovers[takeovers.Username == who].iloc[0]
    n = len(logs[logs.Username == who])
    print(f"  {who}   {r['Login Timestamp']:%d %b %H:%M}   from {r['Country']}   "
          f"({n} rows in total)")
print()
print('Try it:   logs[logs.Username == "user_840bfc"].sort_values("Login Timestamp")')
print()
print(f"The other {len(takeovers) - 3} were added for this lab, so that 3 examples did not")
print("have to carry the whole exercise. You will meet them at STEP 8, after you")
print("have scored your own detector. Do not go looking for them before then -")
print("the whole point is to find them WITHOUT the answer key.")

---
## STEP 4 — How much did yesterday's label actually know?

Before we throw the label away, let us see how much of the truth it had.

In [ ]:
# ============================================================
# STEP 4 — THE OLD LABEL vs REALITY
# We compare the blocklist column against the three attacks
# we just looked at with our own eyes.
# ============================================================

attack_accounts = ["user_a44feb", "user_fd805b", "user_840bfc", "user_a4549c", "user_c99b98"]
known_bad = logs[logs.Username.isin(attack_accounts)]

print(f"Rows belonging to accounts we KNOW were attacked : {len(known_bad)}")
print(f"...of those, how many did the blocklist flag?    : {known_bad['Is Attack IP'].sum()}")
print(f"...so the blocklist missed                       : "
      f"{1 - known_bad['Is Attack IP'].sum()/len(known_bad):.0%} of them")
print()
print(f"Meanwhile the blocklist flags {logs['Is Attack IP'].sum()} rows in total -")
print("most of which are ordinary people whose address once had a bad neighbour.")
print()
print("THIS is why we are not using it. It is not the truth. It is a rumour.")

---
## STEP 5 — Turn logins into numbers

A model cannot read `"Mozilla/5.0 (Macintosh..."`. It can only do arithmetic. So we
have to turn each login into a short row of numbers. We are building **eight**:

| Feature | Question it answers | Why it might matter |
|---|---|---|
| `fail` | Did this login fail? | 1 = failed, 0 = succeeded |
| `foreign` | Is it from outside Norway? | Where the company is |
| `night` | Was it between midnight and 6am? | Attackers keep odd hours. *Do they, though? We will check.* |
| `bot` | Did the client call itself a bot? | Automation announcing itself |
| `prev_fail_user` | **How many times has this account failed BEFORE now?** | One failure is a typo. Twenty is an attack. |
| `prev_fail_ip` | **How many times has this address failed BEFORE now?** | Catches one machine working through a list |
| `new_country` | Is this a country this account has never used before? | Sudden impossible travel |
| `log_gap` | How long since this account's last login? | Nine seconds apart is not human |

Notice that **three of these depend on history** — on what happened *before* this row.
That is exactly where people blow their own foot off, so we are going to do it the
wrong way first, on purpose.

### 5a — THE WRONG WAY (do not do this at home)

The obvious, one-line way to count how many times each account failed is
`logs.groupby("Username")` over the whole file. It is also completely wrong, and the
reason is worth twenty minutes of anyone's career:

> **It counts failures that have not happened yet.**

If an account fails 30 times on Sunday, this method writes "30" onto that account's
row from **Tuesday** too. Your model then appears to predict Tuesday's attack
brilliantly — because you showed it the future. This is called **data leakage**, and
it is the single most common reason a model that scored 0.99 in testing falls apart
in production.

In [ ]:
# ============================================================
# STEP 5a — THE WRONG WAY, DELIBERATELY
# Count each account's failures across the WHOLE file and stamp
# that total onto every one of its rows. Looks harmless. Isn't.
# ============================================================

leaky = logs.groupby("Username")["Login Successful"].transform(lambda s: (~s).sum())

# Look at the takeover victim. The FIRST row is from June 2nd, six days before
# anything bad happened to this account.
peek = logs[logs.Username == "user_c99b98"].copy()
peek["leaky_fail_count"] = leaky[peek.index]

print(peek[["Login Timestamp", "Login Successful", "leaky_fail_count"]].head(3).to_string(index=False))
print()
print("Look at the June 2nd row. Nothing has gone wrong yet - the real user just")
print("logged in normally. But 'leaky_fail_count' already says 10, because it can")
print("see the attack that happens on June 8th.")
print()
print("A model fed this column is not predicting. It is REMEMBERING THE FUTURE.")

### 5b — THE RIGHT WAY

We walk through the log **in time order, one row at a time**, and for each row we
only ever look at a running tally of what has already happened. When we are done
with a row, *then* we update the tally.

It is a loop instead of a one-liner. It takes a couple of seconds instead of
milliseconds. That is the entire cost of not fooling yourself.

In [ ]:
# ============================================================
# STEP 5b — THE RIGHT WAY: causal (past-only) history counts
#
# Picture a security guard reading the log line by line with a
# notepad. For each line they write down what the notepad says
# RIGHT NOW - then they add this line to the notepad.
# They can never see further down the page.
# ============================================================

fails_by_user = {}    # the notepad: account -> failures so far
fails_by_ip   = {}    # the notepad: address -> failures so far
countries_by_user = {}
last_seen_by_user = {}

prev_fail_user, prev_fail_ip, new_country, gap_seconds = [], [], [], []

for username, ip, country, when, failed in zip(
        logs["Username"], logs["IP Address"], logs["Country"],
        logs["Login Timestamp"], ~logs["Login Successful"]):

    # --- READ the notepad (past only) --------------------------------
    prev_fail_user.append(fails_by_user.get(username, 0))
    prev_fail_ip.append(fails_by_ip.get(ip, 0))

    seen = countries_by_user.setdefault(username, set())
    # 0 for a brand-new account (we have no basis to call it unusual yet)
    new_country.append(0 if (country in seen or not seen) else 1)

    previously = last_seen_by_user.get(username)
    gap_seconds.append((when - previously).total_seconds() if previously else 999_999)

    # --- NOW update the notepad --------------------------------------
    seen.add(country)
    last_seen_by_user[username] = when
    if failed:
        fails_by_user[username] = fails_by_user.get(username, 0) + 1
        fails_by_ip[ip] = fails_by_ip.get(ip, 0) + 1

print("history built for", f"{len(prev_fail_user):,}", "rows - past only, no peeking")

In [ ]:
# ============================================================
# STEP 5c — ASSEMBLE THE FEATURE TABLE
# Eight columns of pure numbers. Nothing else goes in here.
# The two label columns are NOT in this list. That is the point.
# ============================================================

f = pd.DataFrame(index=logs.index)
f["fail"]           = (~logs["Login Successful"]).astype(int)
f["foreign"]        = (logs["Country"] != "NO").astype(int)
f["night"]          = (logs["Login Timestamp"].dt.hour < 6).astype(int)
f["bot"]            = (logs["Device Type"] == "bot").astype(int)
f["prev_fail_user"] = prev_fail_user
f["prev_fail_ip"]   = prev_fail_ip
f["new_country"]    = new_country
# Gaps range from 4 seconds to a week. log1p squashes that huge range into
# something a model can handle, without losing the ordering.
f["log_gap"]        = np.log1p(np.clip(gap_seconds, 0, None))

print(f.head(3).to_string())
print()
print("Sanity check - how often is each history feature even switched on?")
print(f"  rows where the account had failed before : {(f.prev_fail_user > 0).sum():,}")
print(f"  rows where the address had failed before : {(f.prev_fail_ip > 0).sum():,}")
print(f"  rows from a country new to that account  : {f.new_country.sum():,}")
print()
print("STOP AND CHECK: the table above has 8 columns and NO label columns.")

---
## STEP 6 — Detector 1: Isolation Forest

### You already know most of this

Yesterday's Decision Tree asked yes/no questions to sort logins into *attack* and
*benign* — and **somebody had to give it the answers** to learn from.

An Isolation Forest asks yes/no questions too. But:

- the questions are picked **completely at random** (*"is prev_fail_ip above 7?"*), and
- **it is never told any answers.**

All it does is count **how many questions it takes to corner each login on its own.**

> A login that looks like thousands of others hides in the crowd. It takes lots of
> questions to separate it from its neighbours.
>
> A login that is strange gets cornered in three or four questions, because there is
> nothing else near it.

Do that 300 times with 300 different sets of random questions, average how many
questions each login needed, and that average **is** the anomaly score. Few questions
to isolate = weird = high score.

> ⚠ **Not the same as a Random Forest.** A Random Forest is many decision trees
> *voting on an answer they were taught*. An Isolation Forest is never taught an
> answer at all — it only measures how easy each point is to separate.

`contamination=0.02` is us saying *"assume roughly 2% of this file is odd"*. It does
not change the scores or the ranking — it only decides where sklearn draws its
yes/no cut-off. We are going to use the ranking, not the cut-off.

In [ ]:
# ============================================================
# STEP 6 — BUILD, FIT AND SCORE THE ISOLATION FOREST
# n_estimators=300  -> play the game 300 times and average
# random_state=42   -> fix the randomness so YOUR numbers match
#                      the numbers in the slides exactly
# ============================================================

forest = IsolationForest(n_estimators=300, contamination=0.02, random_state=42)
forest.fit(f)                       # <- no labels passed in. There is nothing to pass.

# score_samples() returns a number that is LOW for weird rows.
# We flip the sign so that HIGH = weird, which reads more naturally.
logs["forest_score"] = -forest.score_samples(f)

print("Isolation Forest trained on", f"{len(f):,}", "logins and ZERO labels.")
print()
print("The 5 weirdest logins in the entire file, according to the forest:")
show(logs.nlargest(5, "forest_score"), "TOP 5 BY ISOLATION FOREST")

---
## STEP 7 — Detector 2: your first neural network

### The idea, in one sentence

We build a network shaped like an hourglass, and ask it to **copy its input to its
output** — but force everything through a narrow waist in the middle.

```
   8 numbers  ->  6  ->  3  ->  6  ->  8 numbers
   (a login)      squeeze  waist  expand   (the same login, rebuilt)
```

Three numbers is not enough room to memorise 11,494 different logins. So to do well,
the network is forced to learn the **handful of patterns that most logins follow** —
"Norwegian, succeeded, no history of failures", and so on.

Then comes the trick:

> Feed every login through and compare what comes out to what went in.
> **A login it rebuilds accurately is a login that fits a common pattern.**
> **A login it rebuilds badly is a login unlike anything it learned.** That error is
> our anomaly score.

This shape is called an **autoencoder**. And again: we never tell it what an attack is.

### One thing we must do first: scaling

`log_gap` runs up to about 14. `fail` is 0 or 1. A neural network would treat the big
numbers as more important purely because they are bigger. `StandardScaler` rewrites
every column to have an average of 0 and a spread of 1, so all eight get an equal say.

In [ ]:
# ============================================================
# STEP 7a — SCALE THE FEATURES (required before a neural net)
# After this, every column is centred on 0 with a spread of 1.
# ============================================================

scaler = StandardScaler()
X = scaler.fit_transform(f)          # X is a plain numpy array of 8 columns

print("shape       :", X.shape, "  (rows, features)")
print("column means:", np.round(X.mean(axis=0), 3), "  <- all ~0")
print("column stdev:", np.round(X.std(axis=0), 3), "  <- all ~1")

In [ ]:
# ============================================================
# ============================================================
# HERE WE BUILD THE NEURAL NETWORK MODEL
#
# Nothing is learned in this cell. We are only describing the
# SHAPE of the network - how many layers, how wide each one is.
# Think of it as drawing the empty hourglass before pouring
# anything through it.
#
#   Input   8 numbers   (one login)
#   Dense   6 neurons   squeeze
#   Dense   3 neurons   the waist - the bottleneck
#   Dense   6 neurons   expand
#   Output  8 numbers   the rebuilt login
#
# 'relu' and 'tanh' are activation functions: they let the
# network bend, instead of only drawing straight lines.
# ============================================================
# ============================================================

keras.utils.set_random_seed(42)     # so your numbers match the slides

autoencoder = keras.Sequential([
    keras.layers.Input(shape=(8,)),          # 8 features in
    keras.layers.Dense(6, activation="relu"),   # squeeze
    keras.layers.Dense(3, activation="relu"),   # THE BOTTLENECK
    keras.layers.Dense(6, activation="relu"),   # expand
    keras.layers.Dense(8, activation="linear"), # 8 features back out
])

# 'adam' is the standard method for nudging the weights in the right direction.
# 'mse' (mean squared error) is HOW WRONG we say it is: the average squared
# difference between what went in and what came out. That same number becomes
# our anomaly score later on.
autoencoder.compile(optimizer="adam", loss="mse")

autoencoder.summary()
print()
print("Built, but completely untrained. Right now it outputs nonsense.")

In [ ]:
# ============================================================
# ============================================================
# HERE WE TRAIN THE NEURAL NETWORK MODEL
#
# Read the fit() line carefully:
#
#     autoencoder.fit(X, X, ...)
#              input ^  ^ target
#
# The input and the target are THE SAME THING. We are not
# teaching it to spot attacks. We are teaching it to copy.
# There is no label anywhere in this cell.
#
# epochs=30      -> look at the whole dataset 30 times
# batch_size=256 -> in chunks of 256 rows at a time
# shuffle=True   -> in a different order each time
#
# Takes about 20 seconds on a free Colab CPU.
# ============================================================
# ============================================================

history = autoencoder.fit(
    X, X,
    epochs=30,
    batch_size=256,
    shuffle=True,
    verbose=0,            # quiet; we plot the result instead
)

plt.figure(figsize=(6, 3))
plt.plot(history.history["loss"])
plt.title("Training loss — how badly it rebuilds an average login")
plt.xlabel("epoch"); plt.ylabel("mean squared error")
plt.grid(alpha=.3); plt.tight_layout(); plt.show()

print(f"loss at the start : {history.history['loss'][0]:.4f}")
print(f"loss at the end   : {history.history['loss'][-1]:.4f}")
print()
print("Falling and then flattening out is exactly what you want to see.")
print("It means the network found the common patterns and then ran out of")
print("room in that 3-number waist to learn anything more.")

In [ ]:
# ============================================================
# ============================================================
# HERE WE RUN INFERENCE AGAINST THE NEURAL NETWORK MODEL
#
# 'Inference' just means USING a trained model rather than
# training it. We push all 11,494 logins through the hourglass
# and get 11,494 rebuilt logins back.
#
# Then, for each one, we measure HOW WRONG the rebuild was:
#
#     error = average of (original - rebuilt) squared
#
# Small error  -> the network has seen plenty like this.  NORMAL.
# Large error  -> the network has never seen anything like it. WEIRD.
#
# That error IS the anomaly score. No labels. Ever.
# ============================================================
# ============================================================

rebuilt = autoencoder.predict(X, verbose=0)          # <- inference

reconstruction_error = np.mean((X - rebuilt) ** 2, axis=1)
logs["nn_score"] = reconstruction_error

print("inference complete for", f"{len(logs):,}", "logins")
print()
print(f"typical (median) rebuild error : {np.median(reconstruction_error):.4f}")
print(f"worst rebuild error            : {reconstruction_error.max():.4f}")
print(f"                        ...that is {reconstruction_error.max()/np.median(reconstruction_error):.0f}x the typical row")
print()
show(logs.nlargest(5, "nn_score"), "TOP 5 BY NEURAL NETWORK")

---
## STEP 8 — Open the envelope

Both detectors have now ranked all 11,494 logins from weirdest to most ordinary,
**without ever being told what an attack is.**

Time to find out how they did. `day3_answer_key.csv` lists which rows really were
part of an attack, and which rows are **innocent logins that look guilty** — those
are there on purpose, and they are the interesting part.

*In real life this envelope does not exist. That is the whole reason this technique
matters.*

In [ ]:
# ============================================================
# STEP 8a — LOAD THE ANSWER KEY
# ============================================================

KEY_URL = "https://raw.githubusercontent.com/cyberirishman/5-day-AI-Cyber/main/day3_lab6/day3_answer_key.csv"
key = pd.read_csv(KEY_URL)

print(key.attack_type.value_counts().to_string())
print()

REAL_ATTACKS = ["credential_stuffing", "bot_brute_force", "takeover_attempt",
                "takeover_success", "takeover_success_original"]

attack_rows = set(key[key.attack_type.isin(REAL_ATTACKS)]["index"])
tricky_rows = set(key[key.attack_type.str.startswith("hard_negative")]["index"])

logs["truth_attack"] = logs["index"].isin(attack_rows)
logs["truth_tricky"] = logs["index"].isin(tricky_rows)

print(f"genuine attack rows      : {logs.truth_attack.sum()}  "
      f"({logs.truth_attack.mean():.1%} of the file)")
print(f"innocent-but-guilty-looking: {logs.truth_tricky.sum()}")

In [ ]:
# ============================================================
# STEP 8b — SCORE BOTH DETECTORS
#
# 'top N' = you are an analyst with time to look at N alerts today.
# precision = of the N I looked at, how many were real attacks?
# recall    = of all the real attacks, how many did I find?
# ============================================================

def grade(score_column, n):
    top = logs.nlargest(n, score_column)
    caught = top.truth_attack.sum()
    return {
        "alerts looked at": n,
        "real attacks found": int(caught),
        "precision": f"{caught / n:.1%}",
        "recall": f"{caught / logs.truth_attack.sum():.1%}",
        "takeovers found": f"{int(top['Is Account Takeover'].sum())}/18",
        "false alarms": int(n - caught),
    }

rows = []
for n in (120, 240, 600):
    rows.append({"detector": "Isolation Forest", **grade("forest_score", n)})
    rows.append({"detector": "Neural network",   **grade("nn_score", n)})

print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# ============================================================
# STEP 8c — THE CHART: precision as the alert budget grows
# ============================================================

budgets = range(20, 1201, 20)
prec_forest = [logs.nlargest(n, "forest_score").truth_attack.mean() for n in budgets]
prec_nn     = [logs.nlargest(n, "nn_score").truth_attack.mean()     for n in budgets]

plt.figure(figsize=(8, 4))
plt.plot(list(budgets), prec_forest, label="Isolation Forest", linewidth=2)
plt.plot(list(budgets), prec_nn, label="Neural network (autoencoder)", linewidth=2)
plt.axhline(logs.truth_attack.mean(), linestyle="--", color="grey",
            label="random guessing")
plt.xlabel("alerts an analyst reviews (highest score first)")
plt.ylabel("share that are real attacks")
plt.title("Precision vs alert budget — neither detector saw a single label")
plt.legend(); plt.grid(alpha=.3); plt.ylim(0, 1.05)
plt.tight_layout(); plt.show()

### So what happened?

Two things worth carrying out of the room:

**1. Both detectors beat random guessing by a mile, with no labels at all.** Compare
that with yesterday: the labelled Decision Tree flagged 669 innocent logins to catch
158 attacks — over four false alarms for every real catch, and a €3 proxy defeated it.

**2. The simple model beat the neural network.** Not by a little. Look at the chart.
The 20-line Isolation Forest holds high precision far longer than the network that
took 30 epochs to train.

> **This is the lesson, not a bug.** Neural networks win on images, audio and language
> — data where the raw input is enormous and the patterns are deeply buried. On eight
> tidy columns of numbers there is nothing deep to find, and a simpler method with
> fewer ways to go wrong does better. *"We used a neural network"* is not an argument.
> Measuring is.

And it is not close on the case that matters most — run the next cell.

In [ ]:
# ============================================================
# STEP 9 — THE TAKEOVER QUEUE
# The 18 rows in this file where somebody got into an account
# that was not theirs. Where did each detector rank them?
# ============================================================

takeovers = logs[logs["Is Account Takeover"]].copy()
takeovers["forest_rank"] = logs.forest_score.rank(ascending=False)[takeovers.index].astype(int)
takeovers["nn_rank"]     = logs.nn_score.rank(ascending=False)[takeovers.index].astype(int)

print(takeovers[["Login Timestamp", "Username", "Country",
                 "forest_rank", "nn_rank"]].sort_values("forest_rank").to_string(index=False))
print()
print(f"Best possible rank is 1. Worst is {len(logs):,}.")
print()
print(f"worst forest rank of any takeover : {takeovers.forest_rank.max()}")
print(f"worst neural-net rank             : {takeovers.nn_rank.max()}")
print()
print("EVERY ONE of the 18 takeovers is inside the forest's top "
      f"{takeovers.forest_rank.max()} rows out of {len(logs):,}.")
print("So an analyst working a queue of 300 alerts finds all 18 - including three")
print("that nobody in this room has ever labelled or described to the model.")
print()
print("Compare the two columns. The forest ranks every single takeover higher than")
print("the network does. On this data the simple method is not just cheaper - it is")
print("better at the one thing we care about most.")

---
## STEP 10 — Where this fails. Read this bit twice.

An anomaly detector does not find attacks. **It finds unusual things.** Most unusual
things are not attacks, and that gap is where your credibility goes to die.

The answer key contained some deliberately awkward rows. Let us see what our detector
did with them.

In [ ]:
# ============================================================
# STEP 10 — THE FALSE POSITIVES WE PLANTED
# ============================================================

top600 = logs.nlargest(600, "forest_score")
flagged_tricky = key[key["index"].isin(top600[top600.truth_tricky]["index"])]

print("Innocent logins that the detector flagged anyway:")
print(flagged_tricky.attack_type.value_counts().to_string())
print()

# Pick one of the flagged people and show the moment itself, rather than
# whatever they happened to do last.
forgot = key[key.attack_type == "hard_negative_forgot_password"]
flagged_people = logs[logs["index"].isin(forgot["index"])
                      & logs["index"].isin(top600["index"])].Username
who = sorted(flagged_people.unique())[0]

their_burst = forgot[forgot.Username == who]["index"]
their_logins = logs[logs.Username == who].sort_values("Login Timestamp")
lead_in = their_logins[their_logins["index"] < their_burst.min()].tail(1)

show(pd.concat([lead_in, their_logins[their_logins["index"].isin(their_burst)]]),
     f"FALSE POSITIVE — {who} genuinely forgot their own password")

print("Compare that with the takeover in Step 3c. Honestly: what is different?")
print()
print("  Takeover        : several failures, then a success.")
print("  Forgot password : several failures, then a success.")
print()
print("The difference is the ADDRESS and the DEVICE - home and familiar, versus")
print("foreign and scripted. Our features do capture some of that, which is why")
print("most of these rank below the real attacks. But not all of them.")

### The honest summary

| What it is good at | What it cannot do |
|---|---|
| Finding attacks nobody labelled | Telling you *why* a row is odd |
| Surviving a new attack it has never seen | Distinguishing "forgot my password" from "guessing my password" |
| Running with no ground truth at all | Working on a brand-new account with no history |
| Producing a **ranked queue** for a human | Producing a verdict |

**The one sentence to remember:**

> A detector that has never been told the answer cannot be fooled by the answer
> being wrong — but it also cannot tell you what it found. It ranks. **You** decide.

### How this connects to the rest of the course

- **Day 2** you trusted a label and got a model that knew geography.
- **Day 3 (earlier today)** you broke it with a proxy, then fixed the label and found the blind spot had simply moved.
- **Just now** you dropped the label completely, and got better precision with a plainer model.
- **Day 4** you meet models where you cannot see the features at all — and the questions get harder.

---

### If you have time left over

1. Set `night` to always 0 in the feature table and re-run. How much did it matter? (You will be surprised how little.)
2. Drop `bot`. Which attack disappears from the top of the queue?
3. Change `n_estimators` from 300 to 20. Does the forest get much worse? Why not?
4. Make the bottleneck 6 instead of 3. The network rebuilds everything better — so why does that make it a *worse* detector?